# Historic Candle Features And Labels

Build model-ready candle datasets from raw historic CSVs. Each file is loaded with Dask, transformed with `CandleFeatureBuilder`, labelled with the triple-barrier labeller, and only then saved into `../data/historic_candle_feature`.


In [13]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PACKAGE_SRC = PROJECT_ROOT / "packages" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

PROJECT_ROOT

PosixPath('/Users/akash/PycharmProjects/AlgoTrade')

In [14]:
from tradingbot.core.features import CandleFeatureBuilder
from tradingbot.models.label import TripleBarrierLabeller, build_labelled_feature_dataset

INPUT_DIR = PROJECT_ROOT / "data" / "historic_data"
OUTPUT_DIR = PROJECT_ROOT / "data" / "historic_candle_feature"
FILE_PATTERN = "*.csv"
OUTPUT_SUFFIX = "_features_triple_barrier"
OVERWRITE_EXISTING = True
BLOCKSIZE = "16 MiB"

FEATURES = [
    "timestamp",
    "upper_wick_to_candle_ratio",
    "lower_wick_to_candle_ratio",
    "body_to_candle_ratio",
    "candle_type",
    "candle_color_code",
    "log_volume_zscore",
    "indicators",
]

TRIPLE_BARRIER_UPPER_PROFIT_PCT = 2.0
TRIPLE_BARRIER_LOWER_STOP_PCT = 1.0
TRIPLE_BARRIER_MAX_TIME_HORIZON = 10
TRIPLE_BARRIER_VERTICAL_LABEL = "return"

feature_builder = CandleFeatureBuilder(FEATURES)
labeller = TripleBarrierLabeller(
    upper_profit_barrier=TRIPLE_BARRIER_UPPER_PROFIT_PCT,
    lower_stop_barrier=TRIPLE_BARRIER_LOWER_STOP_PCT,
    max_time_horizon=TRIPLE_BARRIER_MAX_TIME_HORIZON,
    vertical_barrier_label=TRIPLE_BARRIER_VERTICAL_LABEL,
)

EXPECTED_OUTPUT_COLUMNS = feature_builder.output_columns + labeller.output_columns
feature_builder.feature_map


{'timestamp': 0,
 'upper_wick_to_candle_ratio': 1,
 'lower_wick_to_candle_ratio': 2,
 'body_to_candle_ratio': 3,
 'candle_type_standard': 4,
 'candle_type_doji': 5,
 'candle_type_hammer': 6,
 'candle_type_inverted_hammer': 7,
 'candle_type_spinning_top': 8,
 'candle_type_marubozu': 9,
 'candle_color_green': 10,
 'candle_color_red': 11,
 'log_volume_zscore': 12,
 'rsi_7': 13,
 'rsi_14': 14,
 'stochastic_7': 15,
 'stochastic_14': 16,
 'stochastic_rsi_7': 17,
 'stochastic_rsi_14': 18,
 'macd': 19,
 'atr': 20,
 'adx': 21}

In [15]:
sample_candle = {
    "timestamp": 1700000000000,
    "open": 100.0,
    "high": 110.0,      
    "low": 90.0,
    "close": 105.0,
    "volume": 1000.0
}
features = feature_builder.encode(sample_candle)
features

{'timestamp': 1700000000000,
 'upper_wick_to_candle_ratio': -0.25,
 'lower_wick_to_candle_ratio': 0.0,
 'body_to_candle_ratio': -0.25,
 'candle_type_standard': 0,
 'candle_type_doji': 0,
 'candle_type_hammer': 1,
 'candle_type_inverted_hammer': 0,
 'candle_type_spinning_top': 0,
 'candle_type_marubozu': 0,
 'candle_color_green': 1,
 'candle_color_red': 0,
 'log_volume_zscore': None,
 'rsi_7': None,
 'rsi_14': None,
 'stochastic_7': None,
 'stochastic_14': None,
 'stochastic_rsi_7': None,
 'stochastic_rsi_14': None,
 'macd': None,
 'atr': None,
 'adx': None}

In [ ]:
from pathlib import Path
import glob
import pandas as pd
from dask import dataframe as dd

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def output_path_for(csv_file):
    source_path = Path(csv_file)
    return OUTPUT_DIR / f"{source_path.stem}{OUTPUT_SUFFIX}{source_path.suffix}"


def output_has_expected_schema(path):
    if not path.exists():
        return False
    try:
        columns = pd.read_csv(path, nrows=0).columns.tolist()
    except pd.errors.EmptyDataError:
        return False
    return columns == EXPECTED_OUTPUT_COLUMNS


csv_files = sorted(glob.glob(str(INPUT_DIR / FILE_PATTERN)))
for index, csv_file in enumerate(csv_files, start=1):
    output_file = output_path_for(csv_file)
    if (
        output_file.exists()
        and not OVERWRITE_EXISTING
        and output_has_expected_schema(output_file)
    ):
        print(f"[{index}/{len(csv_files)}] Skipping {Path(csv_file).name}; already labelled -> {output_file}")
        continue

    print(f"[{index}/{len(csv_files)}] Building labelled feature dataset for {csv_file}")
    raw_df = dd.read_csv(csv_file, blocksize=BLOCKSIZE)
    labelled_features_df = build_labelled_feature_dataset(
        raw_df,
        feature_builder,
        labeller,
    )
    labelled_features_df.to_csv(output_file, single_file=True, index=False)
    print(f"    saved -> {output_file}")


[1/182] Building labelled feature dataset for /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/ADANIPORTS_15minute_1990-01-01T00-00-00plus05-30_now.csv
    saved -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_candle_feature/ADANIPORTS_15minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.csv
[2/182] Building labelled feature dataset for /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/ADANIPORTS_60minute_1990-01-01T00-00-00plus05-30_now.csv
    saved -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_candle_feature/ADANIPORTS_60minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.csv
[3/182] Building labelled feature dataset for /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/ADANIPORTS_day_1990-01-01T00-00-00plus05-30_now.csv
    saved -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_candle_feature/ADANIPORTS_day_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.csv
[4/182] Building labelled feature dataset